# ✅ Проверка решения: инференс, метрики, устойчивость

Ноутбук проверяет, что всё работает после правок кодировки категорий, **не переобучая** поставляемые модели
(кроме двух последних ячеек, помеченных ⏳ — они переобучают модель и по умолчанию выключены).

Порядок: **`Kernel → Restart & Run All`**. Ячейки идут от быстрых к медленным.

| Ячейка | Что проверяет | Ожидание |
| :--- | :--- | :--- |
| 1 | Окружение и загрузка весов | `Seed: 42`, `Порог: 0.4200`, путь скоринга XGBoost |
| 2 | Инференс и сабмит | md5 совпал, `362 / 1000` positives |
| 3 | Устойчивость к новым/отсутствующим категориям | без ошибок, неизвестный уровень → NaN |
| 4 ⏳ | Holdout-валидация с реальными метками | ROC-AUC ≈ 0.99, F1 ≈ 0.95 |
| 5 ⏳ | Полное переобучение 5-fold | метрики как в README + тот же md5 сабмита |

> Ячейки 4 и 5 обучают модель. Ячейка 5 **перезаписывает `models/`** — запускайте её только в копии проекта.

In [1]:
# --- 1. Окружение и загрузка модели ---
import sys, json, hashlib, platform, warnings

# не мешаем диагностике нашего кода, но прячем шум библиотек
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*eval_set.*')

import numpy as np
import pandas as pd
import xgboost, lightgbm, catboost, sklearn

print('python      :', platform.python_version())
print('xgboost     :', xgboost.__version__)
print('lightgbm    :', lightgbm.__version__)
print('catboost    :', catboost.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas      :', pd.__version__)

from src.models import InsurancePropensityEnsemble

ensemble = InsurancePropensityEnsemble.load('models')
print()
print(f'Модель загружена. Seed: {ensemble.seed}, Порог: {ensemble.optimal_threshold:.4f}')
print('Путь скоринга XGBoost :', ensemble.xgb_scoring_strategy,
      '(native = обычный, codes/repaired = обход повреждённого контейнера категорий)')
print('Уровни категорий      :', {k: len(v) for k, v in ensemble.cat_levels.items()})
print('Метрики OOF из metadata.json:')
m = ensemble.cv_metrics['ensemble']
print(f"  ROC-AUC={m['roc_auc']:.5f}  F1={m['f1_score']:.5f}  Accuracy={m['accuracy']:.5f}")

python      : 3.14.2
xgboost     : 3.4.1
lightgbm    : 4.7.0
catboost    : 1.2.10
scikit-learn: 1.9.1
pandas      : 3.0.6

Модель загружена. Seed: 42, Порог: 0.4200
Путь скоринга XGBoost : native (native = обычный, codes/repaired = обход повреждённого контейнера категорий)
Уровни категорий      : {'gender': 2, 'region': 12, 'family_status': 4, 'education': 4, 'employment_type': 4, 'smartphone_brand': 7}
Метрики OOF из metadata.json:
  ROC-AUC=0.99244  F1=0.95315  Accuracy=0.96900


/workspaces/base-project-for-rucode2026ai-sber-hoose-insurance/src/models.py:520: UserWarning: [11:05:04] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  instance.xgboost_models = pickle.load(f)


In [2]:
# --- 2. Инференс на public_test.csv и сверка сабмита ---
import os

SUB = f'submission_seed_{ensemble.seed}.csv'
before = hashlib.md5(open(SUB, 'rb').read()).hexdigest() if os.path.exists(SUB) else None

test_df = pd.read_csv('public_test.csv')
print(f'Размер тестовой выборки: {test_df.shape}')

probs = ensemble.predict_proba(test_df)
preds = (probs >= ensemble.optimal_threshold).astype(int)

submission = pd.DataFrame({'customer_id': test_df['customer_id'], 'accepted': preds})
submission.to_csv(SUB, index=False)

after = hashlib.md5(open(SUB, 'rb').read()).hexdigest()
print(f'\nФайл предсказаний сохранён: {SUB}')
print(f'  md5 до  : {before}')
print(f'  md5 после: {after}')
print('  СОВПАДАЕТ' if before == after else '  ⚠️ ИЗМЕНИЛСЯ (проверьте причину)')

print('\nРаспределение предсказанных классов:')
print(submission['accepted'].value_counts(normalize=True).to_string())
print(f'Положительных: {int(preds.sum())} / {len(preds)}   (ожидается 362 / 1000)')
print(f'Вероятности: min={probs.min():.4f} mean={probs.mean():.4f} max={probs.max():.4f}')

# проверка формата
sample = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample.columns), 'неверные колонки'
assert submission['customer_id'].tolist() == sample['customer_id'].tolist(), 'неверный порядок customer_id'
assert submission['accepted'].isin([0, 1]).all(), 'accepted не 0/1'
print('\nФормат сабмита: OK (колонки, порядок id, значения 0/1)')
submission.head(10)

Размер тестовой выборки: (1000, 28)

Файл предсказаний сохранён: submission_seed_42.csv
  md5 до  : f0019a4d2292d7434ac936a15a653a6c
  md5 после: f0019a4d2292d7434ac936a15a653a6c
  СОВПАДАЕТ

Распределение предсказанных классов:
accepted
0    0.638
1    0.362
Положительных: 362 / 1000   (ожидается 362 / 1000)
Вероятности: min=0.0001 mean=0.3512 max=0.9998

Формат сабмита: OK (колонки, порядок id, значения 0/1)


/workspaces/base-project-for-rucode2026ai-sber-hoose-insurance/src/models.py:349: RuntimeWarning: XGBoost models could not be scored through the regular path (XGBoostError: [11:05:05] /__w/xgboost/xgboost/src/data/../data/cat_container.h:29: Found a category not in the training set for the 2th (0-based) column: `Екатеринбург`
Stack trace:
  [bt] (0) /usr/local/python/3.14.2/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x21ec5c) [0x7c98e961ec5c]
  [bt] (1) /usr/local/python/3.14.2/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x58ea60) [0x7c98e998ea60]
  [bt] (2) /usr/local/python/3.14.2/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x58eb01) [0x7c98e998eb01]
  [bt] (3) /usr/local/python/3.14.2/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x5aeba9) [0x7c98e99aeba9]
  [bt] (4) /usr/local/python/3.14.2/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x813caf) [0x7c98e9c13caf]
  [bt] (5) /usr/local/python/3.14.2/lib/python3.14/site-packages

,customer_id,accepted
0,C106000,0
1,C106001,1
2,C106002,0
3,C106003,0
4,C106004,0
5,C106005,0
6,C106006,0
7,C106007,0
8,C106008,1
9,C106009,0


In [3]:
# --- 3. Устойчивость к новым и отсутствующим категориям ---
from src.features import engineer_features, CAT_COLS

# 3.1 уровни тестовой матрицы совпадают с обучающими
feat, _ = engineer_features(test_df, group_stats=ensemble.group_stats)
aligned = ensemble._aligned_categories(feat[ensemble.feature_names].copy())
for col in CAT_COLS:
    ok = list(aligned[col].cat.categories) == list(ensemble.cat_levels[col])
    print(f'{col:18s} уровни совпадают с train: {ok}')

# 3.2 неизвестный уровень ('retired' есть в public_test, но нет в train) уходит в NaN
unk = test_df['employment_type'] == 'retired'
print(f"\nemployment_type='retired': строк={int(unk.sum())}, код после приведения="
      f"{aligned.loc[unk, 'employment_type'].cat.codes.unique().tolist()}  (-1 = пропуск)")

# 3.3 батч с новым регионом и выброшенным уровнем не роняет скоринг
odd = test_df.copy()
odd.loc[odd['region'] == 'Омск', 'region'] = 'Новый Город'          # новая категория
odd = odd[odd['smartphone_brand'] != 'Google'].reset_index(drop=True)  # отсутствующий уровень
p = ensemble.predict_proba(odd)
print(f'\nБатч с новой/отсутствующей категорией: n={len(p)}, '
      f'вероятности в [0,1]: {bool(np.isfinite(p).all() and p.min() >= 0 and p.max() <= 1)}')

gender             уровни совпадают с train: True
region             уровни совпадают с train: True
family_status      уровни совпадают с train: True
education          уровни совпадают с train: True
employment_type    уровни совпадают с train: True
smartphone_brand   уровни совпадают с train: True

employment_type='retired': строк=1, код после приведения=[-1]  (-1 = пропуск)

Батч с новой/отсутствующей категорией: n=978, вероятности в [0,1]: True


---
## ⏳ Ячейка 4. Holdout-валидация с реальными метками

`public_test.csv` **без меток**, поэтому честные метрики можно получить только на отложенной части
`train.csv`. Здесь модель обучается заново на 80 % данных (3-fold CV, ~1–2 мин) и оценивается на 20 %.
Поставляемые веса из `models/` при этом **не трогаются**.

In [4]:
# --- 4. Holdout-валидация (обучает модель, ~1-2 мин) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score,
                             precision_score, recall_score, confusion_matrix)
from src.utils import find_optimal_threshold

train_df = pd.read_csv('train.csv')
tr, va = train_test_split(train_df, test_size=0.2, stratify=train_df['accepted'], random_state=42)
print(f'train: {tr.shape}, holdout: {va.shape}')

hold = InsurancePropensityEnsemble(n_splits=3, seed=42)
hold.fit(tr)                      # печатает OOF-метрики по фолдам

p_va = hold.predict_proba(va)
y_va = va['accepted'].values

# порог, найденный на holdout, и порог из поставляемой модели
th_va, f1_va = find_optimal_threshold(y_va, p_va)
th_saved = ensemble.optimal_threshold

for name, th in [('порог с holdout', th_va), ('порог из models/', th_saved)]:
    pred = (p_va >= th).astype(int)
    print(f'\n[{name} = {th:.4f}]')
    print(f'  ROC-AUC  : {roc_auc_score(y_va, p_va):.5f}')
    print(f'  F1       : {f1_score(y_va, pred):.5f}')
    print(f'  Accuracy : {accuracy_score(y_va, pred):.5f}')
    print(f'  Precision: {precision_score(y_va, pred):.5f}')
    print(f'  Recall   : {recall_score(y_va, pred):.5f}')

tn, fp, fn, tp = confusion_matrix(y_va, (p_va >= th_saved).astype(int)).ravel()
print(f'\nМатрица ошибок (порог {th_saved:.2f}): TN={tn} FP={fp} FN={fn} TP={tp}')
print('Ожидаемый уровень: ROC-AUC ≈ 0.99, F1 ≈ 0.95 (ниже README на ~0.001-0.005, так как данных меньше)')

train: (4800, 29), holdout: (1200, 29)
=== Starting Training with 3-Fold Stratified CV (Seed=42) ===
Total features: 81
--- Training Fold 1/3 ---
--- Training Fold 2/3 ---
--- Training Fold 3/3 ---

=== Cross-Validation Results ===
CatBoost (Depth 6) : F1=0.95244, AUC=0.99253
CatBoost (Depth 5) : F1=0.95383, AUC=0.99255
LightGBM           : F1=0.94600, AUC=0.99187
XGBoost            : F1=0.94760, AUC=0.99194
Final Ensemble     : F1=0.95220, AUC=0.99270, Accuracy=0.96896, Precision=0.96364, Recall=0.94103
Calibrated Threshold: 0.55

[порог с holdout = 0.3900]
  ROC-AUC  : 0.99099
  F1       : 0.94737
  Accuracy : 0.96500
  Precision: 0.93564
  Recall   : 0.95939

[порог из models/ = 0.4200]
  ROC-AUC  : 0.99099
  F1       : 0.94564
  Accuracy : 0.96417
  Precision: 0.94207
  Recall   : 0.94924

Матрица ошибок (порог 0.42): TN=783 FP=23 FN=20 TP=374
Ожидаемый уровень: ROC-AUC ≈ 0.99, F1 ≈ 0.95 (ниже README на ~0.001-0.005, так как данных меньше)


---
## ⏳ Ячейка 5. Полное переобучение и сверка с README

Полный цикл `train.ipynb`: 5-fold CV на всём `train.csv` (~4 мин) **с сохранением в `models/`**.
Запускайте только в копии проекта — иначе перезапишете релизные артефакты:

```bash
cp -r <папка проекта> /tmp/verify && cd /tmp/verify
```

In [5]:
# --- 5. Полное переобучение (~4 мин) + проверка воспроизводимости ---
if not os.path.exists('.verify_copy_marker'):
    print('Ячейка пропущена: она перезаписывает models/.')
    print('Запуск: cp -r <папка проекта> /tmp/verify && cd /tmp/verify && touch .verify_copy_marker')
else:
    full = InsurancePropensityEnsemble(n_splits=5, seed=42)
    cv = full.fit(pd.read_csv('train.csv'))
    full.save('models')

    print('\n=== Сверка с таблицей из README ===')
    readme = {  # модель: (ROC-AUC, F1)
        'lightgbm': (0.99133, 0.94595), 'xgboost': (0.99174, 0.94939),
        'catboost_d5': (0.99239, 0.95354), 'catboost_d6': (0.99253, 0.95318),
        'ensemble': (0.99244, 0.95315),
    }
    for name, (auc_ref, f1_ref) in readme.items():
        got = cv[name]
        ok = abs(got['roc_auc'] - auc_ref) < 5e-4 and abs(got['f1_score'] - f1_ref) < 5e-4
        print(f"{name:12s} ROC-AUC={got['roc_auc']:.5f} (ref {auc_ref:.5f}) "
              f"F1={got['f1_score']:.5f} (ref {f1_ref:.5f})  {'OK' if ok else 'РАСХОЖДЕНИЕ'}")

    print('\n=== Инференс переобученной моделью ===')
    reloaded = InsurancePropensityEnsemble.load('models')
    p_new = reloaded.predict_proba(pd.read_csv('public_test.csv'))
    sub_new = pd.DataFrame({'customer_id': test_df['customer_id'],
                            'accepted': (p_new >= reloaded.optimal_threshold).astype(int)})
    sub_new.to_csv('submission_retrained.csv', index=False)
    md5_new = hashlib.md5(open('submission_retrained.csv', 'rb').read()).hexdigest()
    print(sub_new['accepted'].value_counts(normalize=True).to_string())
    print('md5 нового сабмита   :', md5_new)
    print('md5 исходного сабмита:', before)
    print('СОВПАДАЕТ' if md5_new == before else 'ОТЛИЧАЕТСЯ (проверьте версии библиотек)')


Ячейка пропущена: она перезаписывает models/.
Запуск: cp -r <папка проекта> /tmp/verify && cd /tmp/verify && touch .verify_copy_marker
